# IC — metadados ambientais (LLaMA 2 + LoRA + RAG) no Google Colab

**Como usar:**
1. Descompacte o `projeto-ic-2.zip` no seu computador e suba a pasta `projeto-ic-2` para o **Meu Drive**.
   Se já existir uma `projeto-ic-2` lá, renomeie a antiga (ex.: `projeto-ic-2-velha`) antes — os dados dela são aproveitados.
2. Abra este notebook (`IC_treino_colab.ipynb`, dentro da pasta) e escolha uma GPU em
   **Ambiente de execução → Alterar o tipo de ambiente de execução**.
3. **Ambiente de execução → Executar tudo.** Ele pede duas coisas: autorizar o Google Drive e o token do Hugging Face.

O notebook:
- acha sozinho a pasta do projeto no Drive;
- traz de outra versão do projeto no seu Drive o que faltar: os JSON-LD, o dataset e o adapter da fase 2 (nada é apagado);
- só gera o dataset e só treina se eles ainda não existirem;
- avalia as 4 combinações (modelo base × fine-tuning, sem e com RAG) e mostra uma tabela no fim.

Se o Colab desconectar no meio do treino: reabra, rode as células 1 a 5 e use a célula **7b (RETOMAR)**.

## 1. Verificar a GPU

In [ ]:
!nvidia-smi

## 2. Conectar o Google Drive
Vai abrir um pop-up pedindo autorização da sua conta Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Encontrar o projeto e trazer os dados
Acha a pasta do projeto no Drive e, se faltar algo, copia de outra versão do projeto que esteja no seu Drive:
os JSON-LD (`data/raw_json`), o dataset da fase 2, o glossário, o adapter treinado e as avaliações antigas.
Nada é apagado nem sobrescrito. Para escolher a pasta na mão, preencha `PROJETO_FORCADO`.

In [ ]:
import glob, os, shutil, subprocess

RAIZ = "/content/drive/MyDrive"
PROJETO_FORCADO = None          # ex.: "/content/drive/MyDrive/projeto-ic-2"


def rodar(cmd):
    """Roda um comando mostrando a saída; interrompe o notebook se falhar."""
    print(">>", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1",            # saída em tempo real
               TQDM_DISABLE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")  # sem barras
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, env=env,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in p.stdout:
        print(linha, end="")
    if p.wait() != 0:
        raise RuntimeError(f"O comando falhou (código {p.returncode}): {cmd}")


def n_json(d):
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if f.endswith(".json") and not f.startswith(".")])


# 1) pasta do projeto: a que tem o código desta versão (src/training/compat.py)
achados = sorted({os.path.realpath(p).rsplit("/src/training/compat.py", 1)[0]
                  for p in glob.glob(f"{RAIZ}/*/src/training/compat.py")
                  + glob.glob(f"{RAIZ}/*/*/src/training/compat.py")})
if PROJETO_FORCADO:
    PROJETO = PROJETO_FORCADO
elif not achados:
    raise RuntimeError("Não achei o projeto no Drive. Descompacte o projeto-ic-2.zip e "
                       "suba a pasta projeto-ic-2 para o Meu Drive.")
else:
    preferidos = [p for p in achados if os.path.basename(p) == "projeto-ic-2"]
    PROJETO = (preferidos or achados)[0]
    if len(achados) > 1:
        print("Versões do código encontradas:", achados)
os.chdir(PROJETO)
print("Projeto:", PROJETO)

# 2) outras versões do projeto no Drive (de onde trazer o que faltar)
outras = sorted({os.path.realpath(os.path.dirname(d))
                 for d in glob.glob(f"{RAIZ}/*/data") + glob.glob(f"{RAIZ}/*/*/data")}
                - {os.path.realpath(PROJETO)},
                key=lambda p: -n_json(f"{p}/data/raw_json"))


def trazer_arquivo(rel):
    for pasta in outras:
        if os.path.isfile(f"{pasta}/{rel}"):
            os.makedirs(os.path.dirname(rel), exist_ok=True)
            shutil.copy2(f"{pasta}/{rel}", rel)
            print(f"  copiado: {rel}  (de {pasta})")
            return


if n_json("data/raw_json") == 0 and outras and n_json(f"{outras[0]}/data/raw_json"):
    fonte = f"{outras[0]}/data/raw_json"
    print(f"Copiando {n_json(fonte)} JSON-LD de {fonte} (leva alguns minutos)...")
    os.makedirs("data/raw_json", exist_ok=True)
    for f in os.listdir(fonte):
        if f.endswith(".json") and not os.path.exists(f"data/raw_json/{f}"):
            shutil.copy2(f"{fonte}/{f}", f"data/raw_json/{f}")
for rel in ["data/processed/dataset_v3.jsonl", "data/glossary/glossary_full.json"]:
    if not os.path.exists(rel):
        trazer_arquivo(rel)
if not os.path.exists("checkpoints/exp01/adapter_config.json"):
    for pasta in outras:   # só o adapter final; os checkpoint-* intermediários ficam
        if os.path.isfile(f"{pasta}/checkpoints/exp01/adapter_config.json"):
            shutil.copytree(f"{pasta}/checkpoints/exp01", "checkpoints/exp01",
                            dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
            print(f"  copiado: checkpoints/exp01  (de {pasta})")
            break
for pasta in outras:      # avaliações antigas (para o rescore)
    for f in glob.glob(f"{pasta}/benchmarks/results/eval_*.json"):
        dst = f"benchmarks/results/{os.path.basename(f)}"
        if not os.path.exists(dst):
            os.makedirs("benchmarks/results", exist_ok=True)
            shutil.copy2(f, dst)

# 3) situação
print()
print("JSON-LD em data/raw_json:", n_json("data/raw_json"))
for rel in ["data/processed/dataset_v3.jsonl", "data/glossary/glossary_full.json",
            "checkpoints/exp01/adapter_config.json"]:
    print(f"{rel}: {'sim' if os.path.exists(rel) else 'não (será gerado)'}")
if n_json("data/raw_json") == 0:
    raise RuntimeError("Nenhum JSON-LD no Drive. Copie os arquivos .json do corpus para "
                       f"{PROJETO}/data/raw_json e rode esta célula de novo.")

## 4. Instalar as bibliotecas
O Colab já vem com PyTorch e CUDA. Instalamos o resto, nas versões testadas com o projeto. Leva 1-2 minutos.

In [ ]:
# transformers/peft fixos nas versões testadas (o transformers 5 mudou argumentos
# do treino; o train.py é compatível com 4.x e 5.x)
!pip install -q -U "transformers==5.20.2" "peft==0.21.2" accelerate bitsandbytes datasets sentencepiece sacrebleu rouge-score pyyaml
import torch
print("\nBibliotecas instaladas.")
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

## 5. Login no Hugging Face
O LLaMA 2 exige aceitar a licença e um token:
- licença: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
- token (tipo *Read*): https://huggingface.co/settings/tokens

Dica: salve o token nos **Secrets** do Colab (ícone de chave, nome `HF_TOKEN`) para não precisar colar toda vez.

In [ ]:
from huggingface_hub import login, whoami

try:
    print("Hugging Face: já autenticado como", whoami()["name"])
except Exception:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not token:
        from getpass import getpass
        token = getpass("Cole o token do Hugging Face (tipo Read) e tecle Enter: ")
    login(token=token.strip())
    print("Hugging Face: autenticado como", whoami()["name"])

## 6. Dataset de treino (só se ainda não existir)
Se o `dataset_v3.jsonl` da fase 2 já está na pasta, ele é mantido. Se não, as 4 etapas geram um novo:
estatísticas → glossário → gerador de perguntas → negativos.

In [ ]:
import json
from collections import Counter

if os.path.exists("data/processed/dataset_v3.jsonl"):
    print("dataset_v3.jsonl já existe e foi mantido (para gerar de novo, apague o arquivo e rode esta célula).")
else:
    rodar("python -m src.analytics.compute --in data/raw_json --out data/analytics/stats.json")
    rodar("python -m src.glossary.build --json-dir data/raw_json --out data/glossary/glossary_full.json")
    rodar("python -m src.prompts.generator --config configs/dataset_v3.yaml")
    rodar("python -m src.prompts.add_negatives --dataset data/processed/dataset_v3.jsonl --json-dir data/raw_json --n 2500")

c = Counter(json.loads(l)["meta"]["kind"].split("_")[0]
            for l in open("data/processed/dataset_v3.jsonl", encoding="utf-8"))
print("Total:", sum(c.values()), "| mistura:", dict(c))

## 6b. Auditoria do corpus e serviços oficiais da ARM
A **auditoria** confere o corpus (registros, duplicatas, cobertura dos campos, registros do Brasil).
O **catálogo** consulta os serviços oficiais da ARM (unidades das variáveis, relatórios de qualidade e sítios) —
começa com 20 produtos; para todos, rode `python -m src.collect.arm_extra` (o cache evita repetir consultas).
Se a ARM não responder, o notebook segue em frente.

In [ ]:
rodar("python -m src.collect.audit")
try:
    rodar("python -m src.collect.arm_extra --max-classes 20")
except RuntimeError as e:
    print("Catálogo da ARM não atualizado agora (o resto do notebook não depende dele):", e)

## 7. Treinar (só se ainda não houver um adapter)
Se o adapter da fase 2 já está em `checkpoints/exp01`, o treino é pulado e as avaliações usam esse modelo.
Para treinar de novo mesmo assim, mude `FORCAR_TREINO` para `True`. Os checkpoints ficam no Drive a cada 500 passos
e o log mostra o progresso com o tempo restante.

In [ ]:
FORCAR_TREINO = False

if os.path.exists("checkpoints/exp01/adapter_config.json") and not FORCAR_TREINO:
    print("Adapter já existe em checkpoints/exp01 — treino pulado.")
else:
    rodar("python -m src.training.train --config configs/exp01_colab.yaml")

## 7b. RETOMAR (só se o Colab caiu no meio do treino)
Rode as células 1 a 5, mude `RETOMAR` para `True` e rode esta. Ela continua do último checkpoint salvo no Drive.

In [ ]:
RETOMAR = False

if RETOMAR:
    rodar("python -m src.training.train --config configs/exp01_colab.yaml --resume")
else:
    print("Nada a retomar (RETOMAR = False).")

## 8. Avaliar — as 4 combinações
Modelo base e fine-tuning, cada um **sem** e **com** RAG, no benchmark `gold_qa.jsonl`.
O índice do RAG é montado sem o catálogo novo da ARM (`--extra none`), para a comparação ficar limpa.
No fim, as avaliações antigas são re-pontuadas com a detecção de recusa corrigida.

In [ ]:
AVAL = "python -m src.eval.run_eval --config configs/exp01_colab.yaml --benchmark benchmarks/gold_qa.jsonl"

rodar("python -m src.rag.build_index --extra none")
rodar(AVAL + " --baseline")
rodar(AVAL + " --baseline --rag")
if os.path.exists("checkpoints/exp01/adapter_config.json"):
    rodar(AVAL)
    rodar(AVAL + " --rag")
else:
    print("Sem adapter em checkpoints/exp01: avaliações do fine-tuning puladas.")
rodar('python -m src.eval.rescore "benchmarks/results/eval_*.json"')

### Resumo das avaliações
O resultado mais recente de cada combinação.

In [ ]:
import glob, json, os

modos = [("modelo base", "baseline"), ("base + RAG", "baseline_rag"),
         ("fine-tuning", "finetuned"), ("fine-tuning + RAG", "finetuned_rag")]
colunas = ["token_f1", "contains_gold", "exact_match", "refusal_accuracy",
           "false_refusal_rate", "rouge_l", "latency_mean_s"]
print(f"{'':20s}" + "".join(f"{c:>20s}" for c in colunas))
for nome, sufixo in modos:
    arqs = [a for a in glob.glob(f"benchmarks/results/eval_*_{sufixo}_2*.json")
            if not a.endswith("_rescored.json")]
    if not arqs:
        print(f"{nome:20s}  (sem resultado)")
        continue
    m = json.load(open(max(arqs, key=os.path.getmtime), encoding="utf-8"))["metrics"]
    def fmt(v):
        return f"{v:20.3f}" if isinstance(v, (int, float)) else f"{'-':>20s}"
    print(f"{nome:20s}" + "".join(fmt(m.get(c)) for c in colunas))

## 8b. A busca acha a ficha certa? (Recall@k, sem GPU)
Mede, por tipo de pergunta, se a ficha correta aparece entre as primeiras devolvidas pela busca.

In [ ]:
rodar("python -m src.rag.eval_retrieval --index data/rag/index.json --dataset data/processed/dataset_v3.jsonl")

## 9. Conversar com o modelo (teste rápido)
Carrega o modelo neste notebook e faz algumas perguntas.

In [ ]:
import torch, yaml
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.common.prompts import build_chat_prompt

cfg = yaml.safe_load(open("configs/exp01_colab.yaml", encoding="utf-8"))
m = cfg["model"]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
base = AutoModelForCausalLM.from_pretrained(m["base"], quantization_config=bnb, device_map="auto")
if os.path.exists(os.path.join(m["output_dir"], "adapter_config.json")):
    from peft import PeftModel
    model = PeftModel.from_pretrained(base, m["output_dir"])
    tok = AutoTokenizer.from_pretrained(m["output_dir"])
else:
    print("Sem adapter: conversando com o modelo base.")
    model, tok = base, AutoTokenizer.from_pretrained(m["base"])
model.eval()
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
sistema = cfg["system_prompt"]


def perguntar(q):
    prompt = build_chat_prompt(sistema, [], new_user_message=q)
    ins = tok(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    print("P:", q)
    print("R:", tok.decode(out[0, ins["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    print()


for q in ["O que é mean_doppler_velocity?",
          "Para que serve reflectivity?",
          "Qual a cotação do dólar hoje?",
          "Preciso de dados de umidade em Southern Great Plains. O que uso?"]:
    perguntar(q)

## 9b. Conversar com o modelo usando RAG
Usa o modelo carregado na célula 9. Abaixo de cada resposta aparecem as fichas consultadas.

In [ ]:
from src.rag.pipeline import RagPipeline

rag = RagPipeline.from_config(cfg)
tok.truncation_side = "left"   # se o prompt passar do limite, corta o início, nunca a pergunta


def perguntar_rag(q):
    res = rag.build_prompt(sistema, q, count_tokens=lambda s: len(tok(s)["input_ids"]))
    ins = tok(res.prompt, return_tensors="pt", truncation=True,
              max_length=rag.settings.max_input_tokens).to(model.device)
    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    print("P:", q)
    print("R:", tok.decode(out[0, ins["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    print("Fontes:", "; ".join(f"[{s['n']}] {s['title']}" for s in res.sources) or "nenhuma")
    print()


for q in ["Qual o período de coleta do dataset 1twrmr?",
          "Qual o DOI do dataset 1twrmr?",
          "O que significa a sigla KAZR?",
          "Qual a cotação do dólar hoje?"]:
    perguntar_rag(q)

## 10. (Opcional) Baixar o adapter treinado para o PC
O adapter já está no seu Drive (`checkpoints/exp01`). Para baixar um .zip, mude `BAIXAR` para `True`.

In [ ]:
BAIXAR = False

if BAIXAR:
    shutil.make_archive("/content/adapter_exp01", "zip", os.path.abspath("checkpoints/exp01"))
    from google.colab import files
    files.download("/content/adapter_exp01.zip")